## Test run: 1 million rows in the primary table, 100,000 rows in the foreign key table.

In [4]:
import psycopg2
import time

DATABASE_URL = "dbname=testdb user=testuser password=mypassword host=postgres-test"
TABLE_1_NAME = "proto_2_uuid_primary"
TABLE_2_NAME = "proto_2_uuid_foreign"
TOTAL_ROWS = 1_000_000
TOTAL_FK_ROWS = 100_000
BATCH_SIZE = 100_000
COMMIT_FREQUENCY = 100

conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

print("Creating tables...")
cur.execute(f"DROP TABLE IF EXISTS {TABLE_2_NAME} CASCADE;")
cur.execute(f"DROP TABLE IF EXISTS {TABLE_1_NAME} CASCADE;")

# Create table with BIGSERIAL primary key
cur.execute(f"""
CREATE TABLE {TABLE_1_NAME} (
    id BIGSERIAL PRIMARY KEY,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE {TABLE_2_NAME} (
    id BIGSERIAL PRIMARY KEY,
    id_ref BIGINT NOT NULL REFERENCES {TABLE_1_NAME}(id),
    data INT NOT NULL,
    value FLOAT NOT NULL
);
""")
conn.commit()

# Tune PostgreSQL for bulk inserts
cur.execute("SET maintenance_work_mem = '2GB';")
cur.execute("SET work_mem = '512MB';")
cur.execute("SET synchronous_commit = OFF;")
conn.commit()

print("\n=== Phase 1: Insert Rows ===\n")

rows_inserted = 0
start_time = time.time()
batch_count = 0

try:
    cur.execute("BEGIN;")
    
    while rows_inserted < TOTAL_ROWS:
        # Insert simple rows
        cur.execute(f"""
            INSERT INTO {TABLE_1_NAME} (created_at)
            SELECT CURRENT_TIMESTAMP
            FROM generate_series(1, {BATCH_SIZE});
        """)
        
        rows_inserted += BATCH_SIZE
        batch_count += 1
        
        if batch_count % COMMIT_FREQUENCY == 0:
            conn.commit()
            elapsed = time.time() - start_time
            rate = rows_inserted / elapsed
            remaining = TOTAL_ROWS - rows_inserted
            eta_sec = remaining / rate if rate > 0 else 0
            print(f"Inserted {rows_inserted:,} rows in {elapsed:.2f}s ({rate:.0f} rows/sec) - ETA: {eta_sec/3600:.2f}h")
            cur.execute("BEGIN;")
    
    conn.commit()
    
except Exception as e:
    conn.rollback()
    print(f"Error Phase 1: {e}")
    raise

elapsed_phase1 = time.time() - start_time
rate_phase1 = rows_inserted / elapsed_phase1

print(f"\nPhase 1 Complete:")
print(f"  Total rows: {rows_inserted:,}")
print(f"  Elapsed time: {elapsed_phase1:.2f}s")
print(f"  Throughput: {rate_phase1:.0f} rows/sec")
print(f"  Est. time for 1B: {1_000_000_000 / rate_phase1 / 3600:.2f} hours")

print("\n=== Phase 2: Insert FK Rows (Sequential) ===\n")

rows_inserted_fk = 0
start_time_fk = time.time()
batch_count_fk = 0

try:
    cur.execute("BEGIN;")
    
    while rows_inserted_fk < TOTAL_FK_ROWS:
        # Sample from primary table sequentially
        cur.execute(f"""
            INSERT INTO {TABLE_2_NAME} (id_ref, data, value)
            SELECT 
                id,
                row_number() OVER() % 1000,
                (row_number() OVER() - 1) * 0.5
            FROM (
                SELECT id FROM {TABLE_1_NAME} 
                ORDER BY id
                LIMIT {BATCH_SIZE}
                OFFSET {rows_inserted_fk}
            ) t;
        """)
        
        rows_inserted_fk += BATCH_SIZE
        batch_count_fk += 1
        
        if batch_count_fk % COMMIT_FREQUENCY == 0:
            conn.commit()
            elapsed = time.time() - start_time_fk
            rate = rows_inserted_fk / elapsed
            print(f"Inserted {rows_inserted_fk:,} rows in {elapsed:.2f}s ({rate:.0f} rows/sec)")
            cur.execute("BEGIN;")
    
    conn.commit()
        
    print("Creating index...")
    start_index_time = time.time()
    cur.execute(f"CREATE INDEX idx_id_ref ON {TABLE_2_NAME}(id_ref);")
    conn.commit()
    elapsed_index = time.time() - start_index_time
    print(f"Index created in {elapsed_index:.2f}s")
    conn.commit()
    
except Exception as e:
    conn.rollback()
    print(f"Error Phase 2: {e}")
    raise

elapsed_phase2 = time.time() - start_time_fk
rate_phase2 = rows_inserted_fk / elapsed_phase2 if elapsed_phase2 > 0 else 0

print(f"\nPhase 2 Complete:")
print(f"  Total rows: {rows_inserted_fk:,}")
print(f"  Elapsed time: {elapsed_phase2:.2f}s")
print(f"  Throughput: {rate_phase2:.0f} rows/sec")

total_elapsed = elapsed_phase1 + elapsed_phase2
total_rows = rows_inserted + rows_inserted_fk
overall_rate = total_rows / total_elapsed

print("\n" + "="*60)
print("BENCHMARK SUMMARY")
print("="*60)
print(f"Phase 1 (Insert): {rows_inserted:,} rows in {elapsed_phase1:.2f}s ({rate_phase1:.0f} rows/sec)")
print(f"Phase 2 (Insert): {rows_inserted_fk:,} rows in {elapsed_phase2:.2f}s ({rate_phase2:.0f} rows/sec)")
print(f"\nTotal: {total_rows:,} rows in {total_elapsed:.2f}s ({overall_rate:.0f} rows/sec)")
print(f"Est. time for 1B rows (Phase 1): {1_000_000_000 / rate_phase1 / 3600:.2f} hours")
print("="*60)

cur.close()
conn.close()


Creating tables...

=== Phase 1: Insert Rows ===


Phase 1 Complete:
  Total rows: 1,000,000
  Elapsed time: 1.64s
  Throughput: 609823 rows/sec
  Est. time for 1B: 0.46 hours

=== Phase 2: Insert FK Rows (Sequential) ===

Creating index...
Index created in 0.02s

Phase 2 Complete:
  Total rows: 100,000
  Elapsed time: 0.62s
  Throughput: 162193 rows/sec

BENCHMARK SUMMARY
Phase 1 (Insert): 1,000,000 rows in 1.64s (609823 rows/sec)
Phase 2 (Insert): 100,000 rows in 0.62s (162193 rows/sec)

Total: 1,100,000 rows in 2.26s (487508 rows/sec)
Est. time for 1B rows (Phase 1): 0.46 hours


## Full test run: 1 billion patches in the primary table, 1 billion patches in the foreign key table.